# Buscacompis
Proyecto de Matemáticas Discretas que te recomienda estudiantes con los que formar un grupo de estudio basado un tu perfil.


In [ ]:
import pandas as pd
import networkx as nx
import itertools
import matplotlib.pyplot as plt

## 1. Cargar el dataset
Leemos el CSV. Cada estudiante tiene texto separado por `;` en `materias`, `hobbies` y `horario_disponible`.
Los convertimos en conjuntos con la función set() de Python que es la estructura matemática que necesitamos.

In [ ]:
# Carga del csv con los datos de prueba
df = pd.read_csv("../data/estudiantes.csv", encoding="latin1", skiprows=1)

# Verificación con las primeras filas
df.head()

In [ ]:
def texto_a_conjunto(texto):    #Esta función convierte las columnas con varios valores en un conjunto con estos valores
    return set(texto.split(";"))



# Creamos 3 columnas nuevas como conjuntos en vez de texto
df["materias_conjunto"] = df["materias"].apply(texto_a_conjunto)
df["hobbies_conjunto"] = df["hobbies"].apply(texto_a_conjunto)
df["horario_conjunto"] = df["horario_disponible"].apply(texto_a_conjunto)

# Revisamos cómo quedó un estudiante de ejemplo
df.loc[0, ["nombre", "materias_conjunto", "hobbies_conjunto", "horario_conjunto"]]

## 2. Similitud de Jaccard
Para dos conjuntos A y B:

$$ Jaccard(A, B) = \frac{|A \cap B|}{|A \cup B|} $$

Este cálculo nos dice cuántos elementos tienen en común, sobre el total de elementos distintos entre los dos. Osea, nos dice qué tanta similitud existe entre dos conjuntos. Da un número entre 0 y 1: un 0 significa que no tienen nada en común y un 1 significa que los conjuntos son idénticos.
Usaremos este cálculo para saber que tántas cosas tienen en común dos estudiantes entre sí y, definiendo un umbral de aceptación, decidir si pueden formar un grupo de trabajo.

In [ ]:
def jaccard(conjunto_a, conjunto_b):         # Esta función calcula la similitud de Jaccard entre dos conjuntos.
                                             #Si ambos conjuntos están vacíos, devolvemos 0 para evitar dividir por cero.
    interseccion = conjunto_a & conjunto_b   # elementos en común
    union = conjunto_a | conjunto_b          # elementos distintos entre los dos
    if len(union) == 0:
        return 0.0
    return len(interseccion) / len(union)

# Prueba con dos conjuntos inventados
prueba_a = {"Cálculo I", "Programación I", "Física I"}
prueba_b = {"Cálculo I", "Programación I", "Matemáticas Discretas I"}
print("Jaccard de prueba:", jaccard(prueba_a, prueba_b))

## 3. Ponderación de similitudes
Hacemos una suma ponderada de las similitudes ya que el horario disponible y las materias son más importantes que los hobbies a la hora de asignar un posible compañero de estudio.
Para este caso definimos un peso de 0.5 para la similitud de las materias, 0.3 para la similitud de horarios y 0.2 para la similitud de hobbies, la fórmula utilizada es la siguiente:
$$Suma = (0.5 \cdot Sim\_materias) + (0.3 \cdot Sim\_horarios) + (0.2 \cdot Sim\_hobbies)$$

In [ ]:
def suma_ponderada(estudiante_a, estudiante_b, peso_materias=0.5, peso_horario=0.3, peso_hobbies=0.2): 
#Esta función recibe dos estudianteses y devueve
#una suma ponderada de las compatibilidades de los conjuntos entre 0 y 1.
    sim_materias = jaccard(estudiante_a["materias_conjunto"], estudiante_b["materias_conjunto"])
    sim_horario = jaccard(estudiante_a["horario_conjunto"], estudiante_b["horario_conjunto"])
    sim_hobbies = jaccard(estudiante_a["hobbies_conjunto"], estudiante_b["hobbies_conjunto"])
    return (peso_materias * sim_materias) + (peso_horario * sim_horario) + (peso_hobbies * sim_hobbies)

# Probamos con dos estudiantes
estudiante_1 = df.iloc[0]
estudiante_2 = df.iloc[1]

print(f"Compatibilidad entre {estudiante_1['nombre']} y {estudiante_2['nombre']}: {suma_ponderada(estudiante_1, estudiante_2):.3f}")

In [ ]:
#Probamos la suma ponderada entre varios estudiantes para ver que los números tengan sentido
parejas_prueba = [(0, 1), (0, 5), (7, 10), (0, 19)]
for i, j in parejas_prueba:
    a = df.iloc[i]
    b = df.iloc[j]
    s = suma_ponderada(a, b)
    print(f"{a['nombre']:20s} <-> {b['nombre']:20s}  score = {s:.3f}")

## 4. Construcción del grafo
Usamos `networkx` para representar a los estudiantes como nodos y sus compatibilidades como aristas.
Sólo se creará una arista entre dos estudiantes si su `suma_ponderada` supera un umbral estipulado de una manera que el grafo no tenga ni demasiadas ni muy pocas conexiones.

In [ ]:
def construir_grafo(df, umbral):        # Esta función crea un grafo de compatibilidad entre estudiantes.
                                        # Cada nodo es un estudiante identificado por su id y cada arista representa
                                        # una compatibilidad calculada con (suma_ponderada) por encima del umbral especificado.
    G = nx.Graph()
    # Añadimos todos los estudiantes como nodos, guardamos el nombre como atributo
    for _, estudiante in df.iterrows():
        G.add_node(estudiante["id"], nombre=estudiante["nombre"])
    # Recorremos todas las parejas posibles de estudiantes sin repetir
    for i, j in itertools.combinations(df.index, 2):
        a = df.iloc[i]
        b = df.iloc[j]
        compatibilidad = suma_ponderada(a, b)
        if compatibilidad >= umbral:
            G.add_edge(a["id"], b["id"], weight=compatibilidad)
    return G

In [ ]:
# Probamos distintos umbrales para ver cuál da un grafo mejor balanceado
umbrales_prueba = [0.3, 0.5, 0.6, 0.7]
for i in umbrales_prueba:
    G = construir_grafo(df, i)
    n_nodos = G.number_of_nodes()
    n_aristas = G.number_of_edges()
    densidad = nx.density(G)
    aislados = list(nx.isolates(G))
    print(f"Umbral {i:.1f}: {n_nodos} nodos, {n_aristas} aristas, densidad = {densidad:.3f}, {len(aislados)} estudiantes sin ninguna conexión")

## 5. Dibujo del grafo
Con la pruebas definimos que el umbral de 0,5 es el más adecuado, genera un buen número de conexiones sin llegar a ser un número muy alto y los nodos aislados son poco
Creamos un función para dibujar el grafo con este umbral donde el grosor de cada arista representa qué tan compatibles son dos estudiantes.

In [ ]:
umbral = 0.5
G = construir_grafo(df, umbral)
# Posiciones de los nodos (spring_layout agrupa visualmente a los más conectados)
pos = nx.spring_layout(G, seed=42, k=1.7, iterations=100)  # seed fijo para que el layout no cambie cada vez que se corre
# Etiquetas: usamos el nombre del estudiante en vez del id
etiquetas = nx.get_node_attributes(G, "nombre")
# Grosor de las aristas proporcional al peso (compatibilidad)
pesos = [G[u][v]["weight"] * 4 for u, v in G.edges()]

plt.figure(figsize=(10, 8))
nx.draw_networkx_nodes(G, pos, node_color="lightblue", node_size=800)
nx.draw_networkx_labels(G, pos, labels=etiquetas, font_size=8)
nx.draw_networkx_edges(G, pos, width=pesos, alpha=0.6,connectionstyle="arc3,rad=0.15", arrows=True, arrowsize=1 )
plt.title(f"Grafo de compatibilidad entre estudiantes (umbral = {umbral})")
plt.axis("off")
plt.show()



## 6. Recomendación de compañeros
Ahora con el grafo hecho podemos consultar para cada estudiante cuáles son sus compañeros más compatibles ordenados por compatibilidad de mayor a menor.

In [ ]:
def recomendar_companeros(G, id_estudiante, top_n=3):     # Esta función devuelve nodos más compatibles de un estudiante en el grafo,
                                                          # ordenados de mayor a menor compatibilidad.
    if id_estudiante not in G:
        return []

    vecinos = G[id_estudiante]                            # devuelve diccionario con datos de nodos vecinos {id_vecino: {"weight": suma_ponderada}}
    vecinos_ordenados = sorted(vecinos.items(), key=lambda x: x[1]["weight"], reverse=True)
    resultado = []
    for id_vecino, datos in vecinos_ordenados[:top_n]:
        nombre_vecino = G.nodes[id_vecino]["nombre"]
        resultado.append((nombre_vecino, datos["weight"]))

    return resultado


In [ ]:
# Prueba con un estudiante conectado (Ana Torres, id=1)
id_prueba = df.loc[df["nombre"] == "Ana Torres", "id"].values[0]
nombre_prueba = df.loc[df["id"] == id_prueba, "nombre"].values[0]

print(f"Compañeros recomendados para {nombre_prueba}:")
for nombre_vecino, compatibilidad in recomendar_companeros(G, id_prueba):
    print(f"  - {nombre_vecino} ,compatibilidad: {compatibilidad:.3f}")

In [ ]:
# Prueba con un estudiante aislado
id_aislado = df.loc[df["nombre"] == "Felipe Aguilar", "id"].values[0]
recomendaciones = recomendar_companeros(G, id_aislado)
if recomendaciones:
    print(f"Compañeros recomendados para Felipe Aguilar:")
    for nombre_vecino, compatibilidad in recomendaciones:
        print(f"  - {nombre_vecino} (compatibilidad: {compatibilidad:.3f})")
else:
    print("Felipe Aguilar no tiene compañeros compatibles con el umbral actual.")

In [ ]:
def mostrar_recomendaciones(G, id_estudiante, top_n=3):
    # Esta función imprime en pantalla las recomendaciones de compañeros para un estudiante,
    # usando recomendar_companeros() para obtener los datos.
    nombre_estudiante = G.nodes[id_estudiante]["nombre"]
    recomendaciones = recomendar_companeros(G, id_estudiante, top_n)

    if recomendaciones:
        print(f"Compañeros recomendados para {nombre_estudiante}:")
        for nombre_vecino, compatibilidad in recomendaciones:
            print(f"  - {nombre_vecino} (compatibilidad: {compatibilidad:.3f})")
    else:
        print(f"{nombre_estudiante} no tiene compañeros compatibles con el umbral actual.")
# Prueba con un estudiante conectado
mostrar_recomendaciones(G, id_prueba)
# Prueba con un estudiante aislado
mostrar_recomendaciones(G, id_aislado)

Además del puntaje de compatibilidad sería mostrarle al estudiante que cosas tiene en común con su compañero recomendado, usando directamente la intersección de conjuntos.


In [ ]:
def elementos_en_comun(df, id_a, id_b):
    # Devuelve un diccionario con las materias, hobbies y horarios que comparten
    # dos estudiantes, usando la intersección de conjuntos (&).
    estudiante_a = df.loc[df["id"] == id_a].iloc[0]
    estudiante_b = df.loc[df["id"] == id_b].iloc[0]

    return {
        "materias": estudiante_a["materias_conjunto"] & estudiante_b["materias_conjunto"],
        "hobbies": estudiante_a["hobbies_conjunto"] & estudiante_b["hobbies_conjunto"],
        "horario": estudiante_a["horario_conjunto"] & estudiante_b["horario_conjunto"],
    }
def mostrar_elementos_en_comun(df, id_a, id_b):
    nombre_a = df.loc[df["id"] == id_a, "nombre"].values[0]
    nombre_b = df.loc[df["id"] == id_b, "nombre"].values[0]
    comunes = elementos_en_comun(df, id_a, id_b)
    print(f"Elementos en común entre {nombre_a} y {nombre_b}:")
    for categoria, valores in comunes.items():
        if valores:
            print(f"  {categoria}: {', '.join(valores)}")
        else:
            print(f"  {categoria}: ninguno en común")

# Prueba
mostrar_elementos_en_comun(df, 2, 10)

## 7. Recomendación de grupo de estudio
Como ya definimos cómo escontrar compañeros compatibles ahora encontraremos un grupo de estudio completo.


In [ ]:
def encontrar_grupos_estudio(G, tamano_minimo=3):    # Esta función busca cliques en el grafo: subconjuntos de estudiantes donde
                                                     # todos son compatibles entre sí.
    cliques = list(nx.find_cliques(G))
    grupos_validos = [c for c in cliques if len(c) >= tamano_minimo]
    return grupos_validos
def mostrar_grupos_estudio(G, tamano_minimo=3):     #Esta función muestra los estudiantes que pueden formar un grupo
    grupos = encontrar_grupos_estudio(G, tamano_minimo)
    if not grupos:
        print(f"No se encontraron grupos de {tamano_minimo} o más estudiantes mutuamente compatibles.")
        return
    for i, grupo in enumerate(grupos, start=1):
        nombres = [G.nodes[id_est]["nombre"] for id_est in grupo]
        print(f"Grupo {i}: {', '.join(nombres)}")


mostrar_grupos_estudio(G, tamano_minimo=3)
mostrar_grupos_estudio(G, tamano_minimo=4)


## 8. Funciones complementarias para un análisis matemático


### Componentes conexas
Una componente conexa es un subconjunto de estudiantes donde todos están conectados entre sí por alguna cadena de compatibilidades, aunque no directamente. Esto identifica los grupos aislados o islas de estudiantes que podrían formar comunidad de estudio separadas del resto.

In [ ]:
def mostrar_componentes_conexas(G):
    componentes = list(nx.connected_components(G))
    print(f"Se encontraron {len(componentes)} componentes conexas:\n")
    for i, componente in enumerate(componentes, start=1):
        nombres = [G.nodes[id_est]["nombre"] for id_est in componente]
        print(f"Componente {i} ({len(nombres)} estudiantes): {', '.join(nombres)}")

mostrar_componentes_conexas(G)

### Grado de los nodos
El grado de un estudiante es su número de conexiones compatibles, por lo que un grado alto indica un estudiante compatible con muchas personas.

In [ ]:
def mostrar_grados(G):
    grados = dict(G.degree())
    grados_ordenados = sorted(grados.items(), key=lambda x: x[1], reverse=True)

    print("Estudiantes ordenados por número de conexiones (grado):")
    for id_est, grado in grados_ordenados:
        nombre = G.nodes[id_est]["nombre"]
        print(f"  {nombre}: {grado} conexiones")

mostrar_grados(G)

### Propiedades de la relación de compatibilidad
Analizamos si "ser compatible" cumple propiedades formales de una relación matemática:
**Simetría**: si A es compatible con B, ¿entonces B es compatible con A?.
**Transitividad**: si A es compatible con B, y B con C, ¿entonces A es compatible con C?.

In [ ]:
def verificar_simetria(G):       # La simetría está garantizada matemáticamente porque G es un grafo no dirigido
                                # igual lo confirmamos de todas formas.
    for a, b in G.edges():
        if not G.has_edge(b, a):
            return False
    return True

def encontrar_no_transitividad(G, limite_ejemplos=5):      # Busca ejemplos donde A-B y B-C están conectados, pero A-C no lo está,
                                                           # ccon esto se demostraría que la relación de compatibilidad no es transitiva.
    ejemplos = []
    for b in G.nodes():
        vecinos_b = list(G.neighbors(b))
        for a, c in itertools.combinations(vecinos_b, 2):
            if not G.has_edge(a, c):
                ejemplos.append((a, b, c))
                if len(ejemplos) >= limite_ejemplos:
                    return ejemplos
    return ejemplos

print("¿La relación es simétrica?", verificar_simetria(G))

print("\nEjemplos de no transitividad (A-B y B-C compatibles, pero A-C no):")
for a, b, c in encontrar_no_transitividad(G):
    nombre_a = G.nodes[a]["nombre"]
    nombre_b = G.nodes[b]["nombre"]
    nombre_c = G.nodes[c]["nombre"]
    print(f"  {nombre_a} -- {nombre_b} -- {nombre_c}  (pero {nombre_a} y {nombre_c} no son compatibles)")
